# 13 · Sequence-to-Sequence Learning and Attention Mechanisms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/13-seq2seq-attention/13_seq2seq_attention.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-13-seq2seq-attention.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 13](https://github.com/anuxlab/deep-learning-systems/wiki/Module-13) |
| **Level** | Intermediate |
| **Typical CPU runtime** | ~2 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 12 |

## What you will do

- Compute additive and dot-product attention by hand and verify weights sum to 1.
- Build a GRU encoder–decoder with teacher forcing and greedy / beam-search decoding.
- Show the fixed-context bottleneck: accuracy vs sequence length with and without attention.
- Visualise the learned alignment matrix (reversal task → anti-diagonal).

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · Attention by hand
Given a decoder state (query) $q$ and encoder states $h_1..h_S$:

1. **scores** $e_i=\mathrm{score}(q,h_i)$ — *dot*: $q^\top h_i$; *scaled dot*: $q^\top h_i/\sqrt d$; *additive (Bahdanau)*: $v^\top\tanh(W_qq+W_hh_i)$;
2. **weights** $\alpha=\mathrm{softmax}(e)$ (a distribution over source positions);
3. **context** $c=\sum_i\alpha_ih_i$.

A tiny numeric example with 3 source states in 2-D so you can verify it with a calculator.

In [ ]:
H = np.array([[1., 0.], [0., 1.], [1., 1.]]); q = np.array([2., 0.])
e = H @ q; a = np.exp(e - e.max()); a /= a.sum(); c = a @ H
print("scores e =", e); print("weights α =", a.round(4), " sum =", a.sum()); print("context c =", c.round(4))
check(np.isclose(a.sum(), 1) and a.argmax() in (0, 2), "weights form a distribution, peaking on keys aligned with q")
check(np.allclose(e, [2, 0, 2]) and np.allclose(a[0], a[2]), "hand-calculable: scores (2,0,2) -> equal weights on states 1 and 3")
rng = np.random.default_rng(0); d = 16; Hs = rng.normal(size=(5, d)); qs = rng.normal(size=d) * 3
sc = lambda s: (lambda e_: np.exp(e_ - e_.max()) / np.exp(e_ - e_.max()).sum())(s)
print("max weight, unscaled dot:", sc(Hs @ qs).max().round(3), "| scaled dot:", sc(Hs @ qs / np.sqrt(d)).max().round(3))
check(sc(Hs @ qs).max() >= sc(Hs @ qs / np.sqrt(d)).max(), "scaling by 1/√d softens an over-peaked softmax")

## 2 · A toy "translation" task: sequence reversal
Variable-length digit sequences (length 3–`Lmax`) must be output reversed. Reversal is a perfect testbed: the correct alignment is the **anti-diagonal**, and long sequences stress the fixed-size context vector of a plain encoder–decoder. Tokens: 0=PAD, 1=BOS, 2=EOS, 3..12 = digits.

In [ ]:
PAD, BOS, EOS, OFF = 0, 1, 2, 3; V = 13
def gen(B, Lmin, Lmax, rng):
    L = rng.integers(Lmin, Lmax + 1, B); src = np.full((B, Lmax), PAD); tgt = np.full((B, Lmax + 1), PAD); tin = np.full((B, Lmax + 1), PAD)
    for i, l in enumerate(L):
        s = rng.integers(OFF, V, l); src[i, :l] = s; r_ = s[::-1]; tgt[i, :l] = r_; tgt[i, l] = EOS; tin[i, 0] = BOS; tin[i, 1:l + 1] = r_
    return [torch.tensor(a) for a in (src, tin, tgt)]
s_, ti_, tg_ = gen(3, 3, 8, np.random.default_rng(0)); print("src:\n", s_.numpy(), "\ndecoder input (BOS+target):\n", ti_.numpy(), "\ntarget (+EOS):\n", tg_.numpy())
check((tg_[0][tg_[0] >= OFF].numpy() == s_[0][s_[0] >= OFF].numpy()[::-1]).all(), "target is the reversed source")

## 3 · Encoder–decoder with optional attention
- **Encoder:** embedding → bidirectional GRU, outputs $h_1..h_S$ (padding masked).
- **Decoder:** GRU cell; at each step input = [embedding of previous token ; context vector].
- *No attention*: context = the encoder's final summary only (fixed bottleneck). *With attention*: context recomputed from all encoder states every step (additive or dot-product).
- **Teacher forcing** at training time (feed ground truth previous token); **greedy** or **beam** decoding at inference.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(s, attn="none", E=32, H=96):
        super().__init__(); s.attn, s.H = attn, H
        s.emb = nn.Embedding(V, E, padding_idx=PAD); s.enc = nn.GRU(E, H // 2, batch_first=True, bidirectional=True)
        s.demb = nn.Embedding(V, E, padding_idx=PAD); s.cell = nn.GRUCell(E + H, H); s.out = nn.Linear(H + H, V); s.init = nn.Linear(H, H)
        if attn == "additive": s.Wq, s.Wk, s.v = nn.Linear(H, H, bias=False), nn.Linear(H, H), nn.Linear(H, 1, bias=False)
        if attn == "dot": s.Wq = nn.Linear(H, H, bias=False)
    def encode(s, src):
        mask = src != PAD; lens = mask.sum(1).cpu(); packed = nn.utils.rnn.pack_padded_sequence(s.emb(src), lens, batch_first=True, enforce_sorted=False)
        o, h = s.enc(packed); o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=src.shape[1])
        summary = torch.cat([h[0], h[1]], -1); return o, mask, summary
    def context(s, q, enc, mask, summary):
        if s.attn == "none": return summary, None
        if s.attn == "additive": e = s.v(torch.tanh(s.Wq(q)[:, None] + s.Wk(enc))).squeeze(-1)
        else: e = (enc @ s.Wq(q)[:, :, None]).squeeze(-1) / s.H ** .5
        a = F.softmax(e.masked_fill(~mask, -1e9), -1); return (a[:, :, None] * enc).sum(1), a
    def step(s, tok, h, enc, mask, summary):
        ctx, a = s.context(h, enc, mask, summary); h = s.cell(torch.cat([s.demb(tok), ctx], -1), h)
        ctx2, a = s.context(h, enc, mask, summary); return s.out(torch.cat([h, ctx2], -1)), h, a
    def forward(s, src, tin):                         # teacher forcing
        enc, mask, summary = s.encode(src); h = torch.tanh(s.init(summary)); logits = []
        for t in range(tin.shape[1]): lg, h, _ = s.step(tin[:, t], h, enc, mask, summary); logits.append(lg)
        return torch.stack(logits, 1)
    @torch.no_grad()
    def greedy(s, src, maxlen, return_attn=False):
        enc, mask, summary = s.encode(src); h = torch.tanh(s.init(summary)); tok = torch.full((len(src),), BOS); outs, att = [], []
        for _ in range(maxlen): lg, h, a = s.step(tok, h, enc, mask, summary); tok = lg.argmax(-1); outs.append(tok); att.append(a)
        return (torch.stack(outs, 1), att) if return_attn else torch.stack(outs, 1)
def seq_acc(model, Lmin, Lmax, n=500, seed=123):
    src, tin, tgt = gen(n, Lmin, Lmax, np.random.default_rng(seed)); model.eval(); pred = model.greedy(src, tgt.shape[1])
    return ((pred == tgt) | (tgt == PAD)).all(1).float().mean().item()          # exact sequence match
print("model classes ready")

## 4 · Train: no attention vs additive vs dot-product
Curriculum: train on lengths 3–`12`. We then evaluate exact-match accuracy as a function of length, including lengths **longer than seen in training** (13–16), which tests both the bottleneck and extrapolation. (Budget: 1,200 steps ≈ 1 min per model on CPU.)

In [ ]:
def train(attn, steps=1200, Lmax=12, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); m = Seq2Seq(attn); o = torch.optim.Adam(m.parameters(), 3e-3); losses = []
    for it in range(steps):
        src, tin, tgt = gen(64, 3, Lmax, rng); lg = m(src, tin); loss = F.cross_entropy(lg.reshape(-1, V), tgt.reshape(-1), ignore_index=PAD)
        o.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); o.step(); losses.append(loss.item())
        if it % 300 == 0: print(f"  [{attn:8s}] step {it:4d} loss {loss.item():.3f}")
    return m, losses
models, curves = {}, {}
with timer("seq2seq_training"):
    for a in ["none", "additive", "dot"]: models[a], curves[a] = train(a)
bins = [(3, 5), (6, 8), (9, 12), (13, 16)]; accs = {a: [seq_acc(m, lo, hi) for lo, hi in bins] for a, m in models.items()}
for a, v in accs.items(): print(f"{a:9s} exact-match by length bin {bins}: {np.round(v, 3)}")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, c in curves.items(): ax[0].plot(np.convolve(c, np.ones(20) / 20, "valid"), label=a)
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("Training loss (smoothed)")
for a, v in accs.items(): ax[1].plot(range(len(bins)), v, "o-", label=a)
ax[1].set_xticks(range(len(bins))); ax[1].set_xticklabels([f"{lo}-{hi}" for lo, hi in bins]); ax[1].set_xlabel("sequence length (last bin = unseen)"); ax[1].set_ylabel("exact match"); ax[1].legend(); ax[1].set_title("Accuracy vs length")
show("seq2seq_accuracy")
check(accs["additive"][1] > accs["none"][1] and accs["dot"][1] > accs["none"][1], "attention beats the fixed-context model on medium-length sequences")
check(max(accs["additive"][0], accs["dot"][0]) > .8, "attention models are accurate on short sequences")
METRICS["seq2seq_exact_match"] = {a: dict(zip([f"{lo}-{hi}" for lo, hi in bins], v)) for a, v in accs.items()}

## 5 · The alignment matrix
Row $t$ of the attention matrix shows where the decoder looked while emitting token $t$. For reversal, output position $t$ should attend to source position $L-1-t$ — an **anti-diagonal**. We plot it for the additive model and quantify how much attention mass falls on the correct source token.

In [ ]:
m = models["additive"]; m.eval(); src, tin, tgt = gen(1, 8, 8, np.random.default_rng(7)); pred, att = m.greedy(src, 9, True)
A = torch.stack(att, 1)[0].numpy()[:9, :8]; L = 8
plt.imshow(A, cmap="viridis", aspect="auto"); plt.xlabel("source position"); plt.ylabel("output step"); plt.colorbar(label="attention weight")
plt.title("Learned alignment (reversal → anti-diagonal)"); plt.xticks(range(8), src[0].numpy()); plt.yticks(range(9), pred[0].numpy()); show("attention_alignment")
mass = np.mean([A[t, L - 1 - t] for t in range(L)]); print(f"mean attention mass on the correct (anti-diagonal) source token: {mass:.2f}")
check(np.allclose(A.sum(1), 1, atol=1e-4), "each row of the alignment matrix sums to 1")
check(mass > .4, "attention concentrates on the anti-diagonal (≫ uniform 1/8 = 0.125)"); METRICS["alignment_mass"] = mass

## 6 · Greedy vs beam-search decoding
Greedy picks the argmax token each step; **beam search** keeps the $k$ highest-scoring *partial sequences* (sum of log-probs) and can recover from an early locally-greedy mistake. Below is a beam search for our attention model. With $k=1$ it must equal greedy; larger beams never find a *lower* log-probability sequence than greedy.

In [ ]:
@torch.no_grad()
def beam_search(m, src, k=4, maxlen=10):                 # one source sequence (1,S)
    enc, mask, summary = m.encode(src); h0 = torch.tanh(m.init(summary)); beams = [(0.0, [BOS], h0, False)]
    for _ in range(maxlen):
        cand = []
        for sc_, seq, h, done in beams:
            if done: cand.append((sc_, seq, h, True)); continue
            lg, h2, _ = m.step(torch.tensor([seq[-1]]), h, enc, mask, summary); lp = F.log_softmax(lg[0], -1)
            top = lp.topk(k)
            for v_, i_ in zip(top.values.tolist(), top.indices.tolist()): cand.append((sc_ + v_, seq + [i_], h2, i_ == EOS))
        beams = sorted(cand, key=lambda b: -b[0])[:k]
        if all(b[3] for b in beams): break
    return beams
src, tin, tgt = gen(1, 10, 10, np.random.default_rng(11)); g = m.greedy(src, 11)[0].tolist()
b1, b4 = beam_search(m, src, 1, 11)[0], beam_search(m, src, 4, 11)[0]
print("target :", tgt[0].tolist()); print("greedy :", g); print(f"beam k=1 ({b1[0]:.3f}):", b1[1][1:]); print(f"beam k=4 ({b4[0]:.3f}):", b4[1][1:])
check(b1[1][1:] == g[:len(b1[1]) - 1], "beam search with k=1 reproduces greedy decoding")
check(b4[0] >= b1[0] - 1e-6, "k=4 finds a sequence with log-probability ≥ greedy's")

In [ ]:
save_metrics("13", METRICS)
log.info('notebook 13 finished OK')

## Wrap-up

## Exercises (do these next)

1. Train with scheduled sampling (feed the model's own prediction with probability p) and compare exact-match.
2. Replace reversal by sorting digits; what alignment pattern does attention learn now?
3. Add coverage (sum of past attention) to the additive attention and see whether it helps on the unseen-length bin.

## Where to read more

- Wiki chapter: [Module 13](https://github.com/anuxlab/deep-learning-systems/wiki/Module-13) (derivations, worked example, Q&A).
- Sutskever et al. (2014); Cho et al. (2014); Bahdanau et al. (2015); Luong et al. (2015); Xu et al. (2015) *Show, attend and tell*.